In [29]:
# importing necessary packages 
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os 
from functools import reduce
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

#survey_current_df = pd.read_csv('/Users/rfernex/Documents/Education/SciencesPo/Courses/CSS/Projects/Project BoJ (Ongoing)/Data/processed/survey_current_situation_df.csv')
#survey_future_df = pd.read_csv('/Users/rfernex/Documents/Education/SciencesPo/Courses/CSS/Projects/Project BoJ (Ongoing)/Data/processed/survey_future_situation_df.csv')
#survey_past_df = pd.read_csv('/Users/rfernex/Documents/Education/SciencesPo/Courses/CSS/Projects/Project BoJ (Ongoing)/Data/processed/survey_past_situation_df.csv')
#survey_perception_df = pd.read_csv('/Users/rfernex/Documents/Education/SciencesPo/Courses/CSS/Projects/Project BoJ (Ongoing)/Data/processed/survey_perception_rates_df.csv')
#Statements_boj_df = pd.read_csv('/Users/rfernex/Documents/Education/SciencesPo/Courses/CSS/Projects/Project BoJ (Ongoing)/Data/Labeled data/Statements__labeled.csv')



In [2]:
# removes NA columns and creates a copy
survey_current_df_clean = survey_current_df[~survey_current_df.astype(str).isin(['-']).any(axis=1)].copy()
# Converts data to numeric
for col in survey_current_df_clean.columns[1:6]:
    survey_current_df_clean[col] = pd.to_numeric(survey_current_df_clean[col], errors='coerce')
    
display(survey_current_df_clean.info())

<class 'pandas.core.frame.DataFrame'>
Index: 73 entries, 2 to 74
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   month_year            73 non-null     object 
 1   Favorable             73 non-null     float64
 2   Somewhat.favorable    73 non-null     float64
 3   Difficult.to.say      73 non-null     float64
 4   Somewhat.unfavorable  73 non-null     float64
 5   Unfavorable           73 non-null     float64
dtypes: float64(5), object(1)
memory usage: 4.0+ KB


None

In [3]:
display(survey_perception_df.head())

,month_year,Too.low,Appropriate,Too.high
0,2006-06,64.8,22.3,9.9
1,2006-09,64.2,25.4,7.7
2,2006-12,63.5,23.0,10.9
3,2007-03,58.2,28.4,10.5
4,2007-06,60.7,23.2,11.8


In [4]:
# Building weighted sentiment scores                      

#current sentiment
weights_array_current = np.array([2, 1, 0, -1, -2])
survey_current_df_clean["wgt_current"] = np.dot(survey_current_df_clean[["Favorable", "Somewhat.favorable", "Difficult.to.say", "Somewhat.unfavorable", "Unfavorable"]], weights_array_current)

#past sentiment
weights_array_future = np.array([1, 0, -1])
survey_future_df["wgt_future"] = np.dot(survey_future_df[["Will.improve", "Will.remain.the.same", "Will.worsen"]], weights_array_future)

#future sentiment
weights_array_past = np.array([1, 0, -1])
survey_past_df["wgt_past"] = np.dot(survey_past_df[["Have.improved", "Have.remained.the.same", "Have.worsened"]], weights_array_past)


#perception sentiment
weights_array_perception = np.array([-1, 0, 1])
survey_perception_df["wgt_perception"] = np.dot(survey_perception_df[["Too.low", "Appropriate", "Too.high"]], weights_array_perception)

#display(survey_current_df_clean.head())
#display(survey_future_df.head())
#display(survey_past_df.head())
#display(survey_perception_df.head())



In [5]:

survey_results = [survey_current_df_clean[["month_year","wgt_current"]], survey_future_df[["month_year","wgt_future"]], survey_past_df[["month_year","wgt_past"]], survey_perception_df[["month_year","wgt_perception"]]]
merged_survey_results = reduce(lambda left, right: pd.merge(left, right, on='month_year', how='outer'), survey_results).iloc[2:]
#display(merged_survey_results.head())
#display(merged_survey_results.info())


,month_year,wgt_current,wgt_future,wgt_past,wgt_perception
2,2006-12,-40.2,-11.4,-12.2,-52.6
3,2007-03,-42.3,-7.7,-12.1,-47.7
4,2007-06,-39.0,-14.5,-12.0,-48.9
5,2007-09,-57.5,-22.9,-27.4,-44.7
6,2007-12,-73.8,-41.4,-40.7,-46.3


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73 entries, 2 to 74
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   month_year      73 non-null     object 
 1   wgt_current     73 non-null     float64
 2   wgt_future      73 non-null     float64
 3   wgt_past        73 non-null     float64
 4   wgt_perception  73 non-null     float64
dtypes: float64(4), object(1)
memory usage: 3.0+ KB


None

In [6]:

# Create a copy of the dataframe
survey_results_scaled = merged_survey_results.copy()

# Select all columns except month_year for scaling
columns_to_scale = [col for col in merged_survey_results.columns if col != 'month_year']

# Apply scaling only to selected columns
scaler = StandardScaler()
survey_results_scaled[columns_to_scale] = scaler.fit_transform(merged_survey_results[columns_to_scale])

#display(survey_results_scaled.head())


,month_year,wgt_current,wgt_future,wgt_past,wgt_perception
2,2006-12,1.028188,1.006990,1.212883,-1.724966
3,2007-03,0.967818,1.304342,1.217315,-1.141826
4,2007-06,1.062686,0.757857,1.221747,-1.284636
5,2007-09,0.530851,0.082787,0.539235,-0.784802
6,2007-12,0.062261,-1.403973,-0.050208,-0.975215


In [7]:
#display(Statements_boj_df.info())
#display(Statements_boj_df.describe())
#display(Statements_boj_df.head())

,url,text,date,month_year,Uncollateralized interest rate
0,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_1998...,Bank of Japan (For immediate release) The Bank...,1998-01-16,1998-01,stable
1,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_1998...,Bank of Japan (For immediate release) The Bank...,1998-02-13,1998-02,stable
2,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_1998...,Bank of Japan (For immediate release) The Bank...,1998-02-26,1998-02,stable
3,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_1998...,Bank of Japan (For immediate release) The Bank...,1998-03-13,1998-03,stable
4,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_1998...,Bank of Japan (For immediate release) The Bank...,1998-03-26,1998-03,stable


In [8]:
Statements_boj_df['month_year'] = pd.to_datetime(Statements_boj_df['month_year'], format='%Y-%m')

# Sorting and removing duplicates with priority
Statements_boj_short_df = Statements_boj_df.sort_values(
    ['month_year', 'Uncollateralized interest rate'], 
    key=lambda x: x.map({'raised': 1, 'lowered': 1, 'stable': 2}) if x.name == 'Uncollateralized interest rate' else x
).drop_duplicates('month_year', keep='first').copy()

Statements_boj_short_df['month_year'] = Statements_boj_short_df['month_year'].dt.strftime('%Y-%m')
Statements_boj_short_df.drop(columns = ['date'], inplace=True)

#display(Statements_boj_short_df.tail(100))
#display(Statements_boj_short_df.info())




,url,text,month_year,Uncollateralized interest rate
269,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_2013...,Bank of Japan Statement on Monetary Policy At...,2013-10,stable
271,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_2013...,Bank of Japan Statement on Monetary Policy 1....,2013-11,stable
272,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_2013...,Bank of Japan Statement on Monetary Policy 1....,2013-12,stable
273,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_2014...,Bank of Japan Statement on Monetary Policy 1....,2014-01,stable
274,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_2014...,Bank of Japan Statement on Monetary Policy 1....,2014-02,stable
...,...,...,...,...
371,https://www.boj.or.jp/en/mopo/mpmdeci/state_20...,At the Monetary Policy Meeting (MPM) held toda...,2024-06,stable
372,https://www.boj.or.jp/en/mopo/mpmdeci/state_20...,At the Monetary Policy Meeting (MPM) held toda...,2024-07,stable
373,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_2024...,Bank of Japan Statement on Monetary Policy 1....,2024-09,raised
374,https://www.boj.or.jp/en/mopo/mpmdeci/mpr_2024...,Bank of Japan Statement on Monetary Policy At...,2024-10,stable


<class 'pandas.core.frame.DataFrame'>
Index: 289 entries, 0 to 375
Data columns (total 4 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   url                             289 non-null    object
 1   text                            289 non-null    object
 2   month_year                      289 non-null    object
 3   Uncollateralized interest rate  289 non-null    object
dtypes: object(4)
memory usage: 11.3+ KB


None

In [23]:

Statement_and_Survey = [Statements_boj_short_df[["month_year","Uncollateralized interest rate"]],
                        survey_results_scaled[["month_year","wgt_current","wgt_future","wgt_past","wgt_perception"]]]
Statement_and_Survey_df = reduce(lambda left, right: pd.merge(left, right, on='month_year', how='outer'), 
                                 Statement_and_Survey).dropna()

display(Statement_and_Survey_df.head())
display(Statement_and_Survey_df.info())


,month_year,Uncollateralized interest rate,wgt_current,wgt_future,wgt_past,wgt_perception
107,2006-12,stable,1.028188,1.006990,1.212883,-1.724966
110,2007-03,stable,0.967818,1.304342,1.217315,-1.141826
113,2007-06,stable,1.062686,0.757857,1.221747,-1.284636
116,2007-09,stable,0.530851,0.082787,0.539235,-0.784802
119,2007-12,stable,0.062261,-1.403973,-0.050208,-0.975215


<class 'pandas.core.frame.DataFrame'>
Index: 73 entries, 107 to 288
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   month_year                      73 non-null     object 
 1   Uncollateralized interest rate  73 non-null     object 
 2   wgt_current                     73 non-null     float64
 3   wgt_future                      73 non-null     float64
 4   wgt_past                        73 non-null     float64
 5   wgt_perception                  73 non-null     float64
dtypes: float64(4), object(2)
memory usage: 4.0+ KB


None

In [49]:
print(Statement_and_Survey_df['Uncollateralized interest rate'].shape)

X_train, X_test, y_train, y_test = train_test_split(
    Statement_and_Survey_df.drop(['month_year', 'Uncollateralized interest rate'], axis=1), 
    Statement_and_Survey_df['Uncollateralized interest rate'], 
    train_size=0.7, random_state=0, stratify= Statement_and_Survey_df['Uncollateralized interest rate']
)

#print("\
#Training set shape:", X_train.shape)
#print("Testing set shape:", X_test.shape)
#print("\
#Class distribution in training set:")
#print(y_train.value_counts())
#print("\
#Class distribution in test set:")
#print(y_test.value_counts())

(73,)
